# 06. Module C: Risk-Aware RL Battery Trading — Results

Mirrors `05_module_b_evaluation.ipynb` section-by-section.

| Section | Topic |
|---------|-------|
| 1 | Setup & imports |
| 2 | Prepare the environment (Module B forecasts → `BatteryEnv`) |
| 3 | Train / load PPO and SAC agents |
| 4 | Global performance metrics |
| 5 | Episode profit distribution |
| 6 | Dispatch profile — when does the agent trade? |
| 7 | SoC trajectory samples |
| 8 | Risk analysis — CVaR and loss episodes |
| 9 | Naive benchmark comparison |
| 10 | PPO vs SAC final leaderboard |

**Pipeline link:** `BatteryEnv` consumes Module B q10/q50/q90 price forecasts as 72 of its 78 observation dimensions — the uncertainty-propagation hand-off from Module B to Module C.

## 1. Setup

In [ ]:
import os, sys, pickle
from pathlib import Path

import torch
import pyarrow  # must precede pandas to avoid extension-type re-registration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('../src'))

%load_ext autoreload
%autoreload 2
%aimport -gymnasium

from module_c import (
    BatteryEnv, BatteryEnvConfig, CvarRewardShaper,
    PPOAgent, SACAgent, TrainConfig, EvalResult, evaluate_agent,
)
from module_c.reward import compute_cvar

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

CHECKPOINT_DIR = Path('../checkpoints/module_c')
DATA_DIR       = Path('../data/splits')
SEED = 42
print('Environment ready.')

## 2. Prepare the Environment

`BatteryEnv` needs:
- `price_eur_mwh` — actual hourly clearing price
- `q10_h1` ... `q90_h24` — Module B 24-horizon quantile forecasts

Module B's `prepare_supervised` produces a *long* `(origin, horizon)` layout.
The helper below pivots it to *wide* format: one row per origin hour, 72 forecast columns.

In [ ]:
def make_module_b_forecasts(df, model):
    """Wide-format Module B forecasts: columns q10_h1..q90_h24."""
    from module_b import build_features, prepare_supervised, REGISTRY
    from module_b.features import PRICE_COL

    feat      = build_features(df, list(REGISTRY))
    past_cols = [c for c in feat.columns if c != PRICE_COL]
    fut_cols  = [c for c in ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
                              'is_weekend', 'is_holiday_DE']
                 if c in feat.columns]
    X, _ = prepare_supervised(feat, horizons=range(1, 25),
                               past_cols=past_cols, future_cols=fut_cols)
    q = model.predict_quantiles(X)
    q['origin_ts'] = X['origin_ts'].values
    q['horizon_h'] = X['horizon_h'].values

    wide = q.pivot(index='origin_ts', columns='horizon_h',
                   values=['q10', 'q50', 'q90'])
    wide.columns = [f'{qq}_h{h}' for qq, h in wide.columns]
    wide.index.name = None
    return wide


def build_env_df(split_df, fc):

    """Join actual prices with wide forecast columns."""
    return split_df[['price']].join(fc, how='left').ffill().dropna()


def make_synthetic_forecasts(df):
    """Stand-in when Module B checkpoint is unavailable.

    Actual price + Gaussian noise scaled by horizon — mimics real degradation.
    """
    rng = np.random.default_rng(SEED)
    p = df['price'].values
    rows = {}
    for h in range(1, 25):
        shifted = np.roll(p, -h)
        noise   = rng.normal(0, 5.0 + h * 1.5, len(p))
        rows[f'q50_h{h}'] = shifted + noise
        rows[f'q10_h{h}'] = rows[f'q50_h{h}'] - rng.uniform(5, 20, len(p))
        rows[f'q90_h{h}'] = rows[f'q50_h{h}'] + rng.uniform(5, 20, len(p))
    return pd.DataFrame(rows, index=df.index)

In [ ]:
train_raw = pd.read_parquet(DATA_DIR / 'train.parquet')
val_raw   = pd.read_parquet(DATA_DIR / 'val.parquet')
test_raw  = pd.read_parquet(DATA_DIR / 'test.parquet')

print(f'Train: {len(train_raw):,}  Val: {len(val_raw):,}  Test: {len(test_raw):,}')

In [ ]:
from module_b import CatBoostQuantileForecaster, ConformalQuantileRegressor

MB_CHECKPOINT = Path('../checkpoints/module_b')

if MB_CHECKPOINT.exists():
    base_model = CatBoostQuantileForecaster.load(MB_CHECKPOINT)
    cqr_path = MB_CHECKPOINT / 'cqr_state.pkl'
    if cqr_path.exists():
        with cqr_path.open('rb') as fh:
            cqr_state = pickle.load(fh)
        mb_model = ConformalQuantileRegressor(base=base_model, alpha=cqr_state['alpha'])
        mb_model.delta = cqr_state['delta']
    else:
        mb_model = base_model
    print('Module B loaded.')
    train_fc = make_module_b_forecasts(train_raw, mb_model)
    val_fc   = make_module_b_forecasts(val_raw,   mb_model)
    test_fc  = make_module_b_forecasts(test_raw,  mb_model)
else:
    print('WARNING: Module B checkpoint not found — using synthetic price forecasts.')
    train_fc = make_synthetic_forecasts(train_raw)
    val_fc   = make_synthetic_forecasts(val_raw)
    test_fc  = make_synthetic_forecasts(test_raw)

train_env_df = build_env_df(train_raw, train_fc)
val_env_df   = build_env_df(val_raw,   val_fc)
test_env_df  = build_env_df(test_raw,  test_fc)

print(f'Env DataFrames — Train: {len(train_env_df):,}  '
      f'Val: {len(val_env_df):,}  Test: {len(test_env_df):,}')
print(f'Obs columns sample: {list(test_env_df.columns[:4])} '
      f'... ({len(test_env_df.columns)} total)')

In [ ]:
cfg = BatteryEnvConfig(
    capacity_mwh=100.0, max_power_mw=50.0,
    efficiency=0.90,    episode_len=24,
)

train_env = BatteryEnv(
    train_env_df, config=cfg,
    reward_shaper=CvarRewardShaper(lambda_risk=0.1), seed=SEED,
)
val_env = BatteryEnv(
    val_env_df, config=cfg,
    reward_shaper=CvarRewardShaper(lambda_risk=0.1), seed=SEED,
)
test_env = BatteryEnv(
    test_env_df, config=cfg,
    reward_shaper=CvarRewardShaper(lambda_risk=0.1), seed=SEED,
)

print(f'Obs space : {train_env.observation_space}')
print(f'Act space : {train_env.action_space}')

## 3. Train / Load Agents

PPO (on-policy, `n_envs=4`) and SAC (off-policy, replay buffer, `n_envs=1`) both use `lambda_risk=0.1` and `total_timesteps=500_000`.
Checkpoints are loaded automatically when they already exist.

> `eval_freq` / `n_eval_episodes` in `TrainConfig` are stored but not wired to an `EvalCallback` here — add one if you want mid-training curves on `val_env`.

In [ ]:
ppo_path = CHECKPOINT_DIR / 'ppo'
sac_path = CHECKPOINT_DIR / 'sac'

cfg_ppo = TrainConfig(algo='PPO', total_timesteps=500_000,
                      lambda_risk=0.1, n_envs=4, seed=SEED)
cfg_sac = TrainConfig(algo='SAC', total_timesteps=100_000,
                      lambda_risk=0.1, n_envs=1, seed=SEED)

if (ppo_path / 'meta.json').exists():
    ppo_agent = PPOAgent.load(ppo_path)
    print('PPO: loaded from checkpoint.')
else:
    print('PPO: training (may take several minutes)...')
    ppo_agent = PPOAgent()
    ppo_agent.train(train_env, cfg_ppo)
    ppo_agent.save(ppo_path)
    print(f'PPO: saved to {ppo_path}')

if (sac_path / 'meta.json').exists():
    sac_agent = SACAgent.load(sac_path)
    print('SAC: loaded from checkpoint.')
else:
    print('SAC: training (may take several minutes)...')
    sac_agent = SACAgent()
    sac_agent.train(train_env, cfg_sac)
    sac_agent.save(sac_path)
    print(f'SAC: saved to {sac_path}')

## 4. Global Performance Metrics

Mirrors Section 3 of `05_module_b_evaluation.ipynb` — one table, all key numbers.

In [ ]:
N_EVAL = 50

ppo_result = evaluate_agent(ppo_agent, test_env, n_episodes=N_EVAL)
sac_result = evaluate_agent(sac_agent, test_env, n_episodes=N_EVAL)

summary_df = pd.DataFrame([
    {'Agent': 'PPO',
     'Mean Profit (EUR)':   ppo_result.mean_profit,
     'Std Profit (EUR)':    ppo_result.std_profit,
     'Sharpe':              ppo_result.sharpe,
     'CVaR 5% (EUR/step)': ppo_result.mean_cvar,
     'Episodes':            ppo_result.n_episodes},
    {'Agent': 'SAC',
     'Mean Profit (EUR)':   sac_result.mean_profit,
     'Std Profit (EUR)':    sac_result.std_profit,
     'Sharpe':              sac_result.sharpe,
     'CVaR 5% (EUR/step)': sac_result.mean_cvar,
     'Episodes':            sac_result.n_episodes},
]).set_index('Agent')

print('=== Overall Test Metrics ===')
display(summary_df.style.format({
    'Mean Profit (EUR)':   '{:,.0f}',
    'Std Profit (EUR)':    '{:,.0f}',
    'Sharpe':              '{:.3f}',
    'CVaR 5% (EUR/step)': '{:.2f}',
}))

## 5. Episode Profit Distribution

Mirrors Section 4 (horizon-wise analysis). Unit here is the 24-step trading day. Dashed = mean, dotted = 5th percentile.

In [ ]:
def collect_episode_profits(agent, env, n=N_EVAL):
    profits = []
    for _ in range(n):
        obs, _ = env.reset()
        ep, terminated, truncated = 0.0, False, False
        while not (terminated or truncated):
            action = agent.act(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(action)
            ep += info['step_profit']
        profits.append(ep)
    return np.array(profits)

ppo_profits = collect_episode_profits(ppo_agent, test_env)
sac_profits = collect_episode_profits(sac_agent, test_env)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, profits, label, color in zip(
    axes,
    [ppo_profits, sac_profits],
    ['PPO', 'SAC'],
    ['tab:blue', 'tab:orange'],
):
    ax.hist(profits, bins=20, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(profits.mean(), color='black', ls='--',
               label=f'Mean = {profits.mean():,.0f} EUR')
    ax.axvline(np.percentile(profits, 5), color='red', ls=':',
               label=f'P5  = {np.percentile(profits, 5):,.0f} EUR')
    ax.set_title(f'{label} — Episode Profit')
    ax.set_xlabel('Total Episode Profit (EUR)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Episode Profit Distributions — Test Set', y=1.02)
plt.tight_layout()
plt.show()

## 6. Dispatch Profile — When Does the Agent Trade?

Mirrors Section 8 (error by hour of day). Mean power dispatch per hour across all test episodes: green = discharge/sell, red = charge/buy.

In [ ]:
def collect_records(agent, env, n=30):
    records = []
    for ep_idx in range(n):
        obs, _ = env.reset()
        terminated, truncated, step = False, False, 0
        while not (terminated or truncated):
            action = agent.act(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(action)
            records.append({
                'episode':      ep_idx,
                'step':         step,
                'hour':         step % 24,
                'price':        info['price'],
                'power_actual': info['power_mw_actual'],
                'soc_mwh':      info['soc_mwh'],
                'step_profit':  info['step_profit'],
            })
            step += 1
    return pd.DataFrame(records)

ppo_rec = collect_records(ppo_agent, test_env)
sac_rec = collect_records(sac_agent, test_env)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, rec, label in zip(axes, [ppo_rec, sac_rec], ['PPO', 'SAC']):
    by_hour = rec.groupby('hour')['power_actual'].mean()
    colors  = ['tab:green' if v >= 0 else 'tab:red' for v in by_hour.values]
    ax.bar(by_hour.index, by_hour.values, color=colors, alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{label} — Mean Dispatch by Hour')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Mean Power (MW)')
    ax.set_xticks(range(24))

plt.suptitle('Dispatch Profile  (green = discharge / sell,  red = charge / buy)', y=1.02)
plt.tight_layout()
plt.show()

# Mean price by hour for context
mean_price = ppo_rec.groupby('hour')['price'].mean()
plt.figure(figsize=(10, 3))
plt.plot(mean_price.index, mean_price.values, marker='o', color='steelblue')
plt.xticks(range(24))
plt.xlabel('Hour of Day')
plt.ylabel('Mean Price (EUR/MWh)')
plt.title('Mean Market Price by Hour (Test Set) — dispatch timing reference')
plt.tight_layout()
plt.show()

## 7. SoC Trajectory Samples

Mirrors Section 5 (visual sample forecasts). Three representative episodes per agent: price, SoC, and power dispatch.

In [ ]:
N_SAMPLES = 3

for rec, label in [(ppo_rec, 'PPO'), (sac_rec, 'SAC')]:
    ep_ids = rec['episode'].unique()[:N_SAMPLES]
    fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(15, 3 * N_SAMPLES))

    for row_idx, ep in enumerate(ep_ids):
        ep_df = rec[rec['episode'] == ep].reset_index(drop=True)
        hours = ep_df['step'].values

        axes[row_idx, 0].plot(hours, ep_df['price'], marker='o', ms=3, color='steelblue')
        axes[row_idx, 0].set_title(f'Ep {ep} — Price (EUR/MWh)')
        axes[row_idx, 0].set_xlabel('Step (hour)')

        axes[row_idx, 1].step(hours, ep_df['soc_mwh'], where='post', color='forestgreen')
        axes[row_idx, 1].axhline(100, color='gray', ls=':', lw=0.8)
        axes[row_idx, 1].axhline(0,   color='gray', ls=':', lw=0.8)
        axes[row_idx, 1].set_ylim(-5, 110)
        axes[row_idx, 1].set_title('SoC (MWh)')
        axes[row_idx, 1].set_xlabel('Step (hour)')

        bc = ['tab:green' if p >= 0 else 'tab:red' for p in ep_df['power_actual']]
        axes[row_idx, 2].bar(hours, ep_df['power_actual'], color=bc, alpha=0.8)
        axes[row_idx, 2].axhline(0, color='black', lw=0.8)
        axes[row_idx, 2].set_title('Power (MW)  ·  green = discharge')
        axes[row_idx, 2].set_xlabel('Step (hour)')

    plt.suptitle(f'{label} — Episode Trajectories', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 8. Risk Analysis — CVaR and Loss Episodes

Mirrors Section 9 (extreme event / spike analysis). CVaR at multiple confidence levels and per-episode CVaR distribution.

In [ ]:
alphas = [0.01, 0.05, 0.10, 0.20]

risk_rows = []
for label, profits in [('PPO', ppo_profits), ('SAC', sac_profits)]:
    row = {'Agent': label}
    for a in alphas:
        row[f'CVaR {int(a*100)}%'] = compute_cvar(profits, alpha=a)
    row['Loss Episodes'] = int((profits < 0).sum())
    row['Loss Rate']     = f'{(profits < 0).mean():.1%}'
    risk_rows.append(row)

risk_df = pd.DataFrame(risk_rows).set_index('Agent')
print('=== Risk Metrics (Episode Profits) ===')
display(risk_df.style.format({c: '{:,.0f}' for c in risk_df.columns if 'CVaR' in c}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, rec, label, color in zip(
    axes, [ppo_rec, sac_rec], ['PPO', 'SAC'], ['tab:blue', 'tab:orange']
):
    ep_cvars = rec.groupby('episode')['step_profit'].apply(
        lambda x: compute_cvar(x.values, alpha=0.05)
    )
    ax.hist(ep_cvars, bins=20, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(ep_cvars.mean(), color='black', ls='--',
               label=f'Mean CVaR = {ep_cvars.mean():.2f}')
    ax.set_title(f'{label} — Episode CVaR(5%) Distribution')
    ax.set_xlabel('CVaR of per-step profits (EUR)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Per-Episode CVaR(5%) Distribution', y=1.02)
plt.tight_layout()
plt.show()

## 9. Naive Benchmark Comparison

Mirrors Section 6 (comparison across test sets). Rule-based reference: charge fully during hours 00-05 (overnight low), discharge during 08-13 (morning peak). No price forecast used.

In [ ]:
CHARGE_HOURS    = set(range(0, 6))
DISCHARGE_HOURS = set(range(8, 14))

def run_naive(env, n_episodes=N_EVAL):
    profits = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        ep, terminated, truncated, step = 0.0, False, False, 0
        while not (terminated or truncated):
            hour = step % 24
            if   hour in CHARGE_HOURS:    action = np.array([-1.0], dtype=np.float32)
            elif hour in DISCHARGE_HOURS: action = np.array([ 1.0], dtype=np.float32)
            else:                         action = np.array([ 0.0], dtype=np.float32)
            obs, _, terminated, truncated, info = env.step(action)
            ep  += info['step_profit']
            step += 1
        profits.append(ep)
    return np.array(profits)

naive_profits = run_naive(test_env)

benchmark_df = pd.DataFrame([
    {'Strategy': 'Naive (rule-based)',
     'Mean Profit (EUR)': naive_profits.mean(),
     'Std (EUR)':         naive_profits.std(),
     'Sharpe': naive_profits.mean() / naive_profits.std() if naive_profits.std() > 0 else 0.0,
     'CVaR 5% (EUR)':    compute_cvar(naive_profits, 0.05),
     'Loss Rate':         f'{(naive_profits < 0).mean():.1%}'},
    {'Strategy': 'PPO',
     'Mean Profit (EUR)': ppo_result.mean_profit,
     'Std (EUR)':         ppo_result.std_profit,
     'Sharpe':            ppo_result.sharpe,
     'CVaR 5% (EUR)':    ppo_result.mean_cvar,
     'Loss Rate':         f'{(ppo_profits < 0).mean():.1%}'},
    {'Strategy': 'SAC',
     'Mean Profit (EUR)': sac_result.mean_profit,
     'Std (EUR)':         sac_result.std_profit,
     'Sharpe':            sac_result.sharpe,
     'CVaR 5% (EUR)':    sac_result.mean_cvar,
     'Loss Rate':         f'{(sac_profits < 0).mean():.1%}'},
]).set_index('Strategy')

print('=== Benchmark Comparison ===')
display(benchmark_df.style.format({
    'Mean Profit (EUR)': '{:,.0f}',
    'Std (EUR)':         '{:,.0f}',
    'Sharpe':            '{:.3f}',
    'CVaR 5% (EUR)':     '{:,.0f}',
}))

labels = ['Naive', 'PPO', 'SAC']
means  = [naive_profits.mean(), ppo_result.mean_profit, sac_result.mean_profit]
stds   = [naive_profits.std(),  ppo_result.std_profit,  sac_result.std_profit]
colors = ['tab:gray', 'tab:blue', 'tab:orange']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, means, yerr=stds, capsize=6,
              color=colors, alpha=0.85, ecolor='black')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Mean Episode Profit (EUR)')
ax.set_title('Strategy Comparison — Mean Profit +/- 1 Std')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(stds) * 0.05,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 10. PPO vs SAC Final Leaderboard

Mirrors the leaderboard in `B6_final_eval.ipynb`. Higher Sharpe = better risk-adjusted return. CVaR closer to 0 = better tail risk control.

In [ ]:
leaderboard = pd.DataFrame([
    {'Agent': 'PPO',
     'Mean Profit (EUR)': ppo_result.mean_profit,
     'Std (EUR)':         ppo_result.std_profit,
     'Sharpe':            ppo_result.sharpe,
     'CVaR 5%':           ppo_result.mean_cvar,
     'Loss Rate':         f'{(ppo_profits < 0).mean():.1%}'},
    {'Agent': 'SAC',
     'Mean Profit (EUR)': sac_result.mean_profit,
     'Std (EUR)':         sac_result.std_profit,
     'Sharpe':            sac_result.sharpe,
     'CVaR 5%':           sac_result.mean_cvar,
     'Loss Rate':         f'{(sac_profits < 0).mean():.1%}'},
]).set_index('Agent')

print('=== Module C Final Leaderboard (Test — 2025 Q1) ===')
display(
    leaderboard.style
    .format({
        'Mean Profit (EUR)': '{:,.0f}',
        'Std (EUR)':         '{:,.0f}',
        'Sharpe':            '{:.3f}',
        'CVaR 5%':           '{:.2f}',
    })
    .highlight_max(subset=['Mean Profit (EUR)', 'Sharpe'], color='lightgreen')
    .highlight_min(subset=['CVaR 5%'],                    color='lightyellow')
)

best = leaderboard['Sharpe'].idxmax()
print(f'\nProduction recommendation: {best} '
      f'(highest Sharpe on locked 2025 Q1 test set).')